IMPORT FILE

In [ ]:
#Basic Libraries
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import itertools

sb.set()


climateData = pd.read_excel("climate_change_download_0 (1).xls", sheet_name = "Data")
climateData


In [ ]:

#variables to keep
keep_series = {
    "Cereal yield (kg per hectare)",
    "Foreign direct investment, net inflows (% of GDP)",
    "Urban population",
    "Paved roads (% of total roads)",
    "CO2 emissions, total (KtCO2)",
    "Energy use per capita (kilograms of oil equivalent)",
    "Energy use per units of GDP (kg oil eq./$1,000 of 2005 PPP $)",
    "GHG net emissions/removals by LUCF (MtCO2e)",
    "Population in urban agglomerations >1million (%)",
    "Nationally terrestrial protected areas (% of total land area)",
    "GDP ($)",
    "GNI per capita (Atlas $)",
    "Under-five mortality rate (per 1,000)",
    "Population growth (annual %)",
    "Population",
    "Urban population growth (annual %)",
}

df_filtered = climateData[climateData["Series name"].astype(str).str.strip().isin(keep_series)].copy()
cleanData = pd.DataFrame(df_filtered)

cleanData

In [ ]:
cols_to_drop = ["Country code", "Series code", "SCALE", "Decimals"]
cleanData = cleanData.drop(columns=cols_to_drop, errors="ignore")
cleanData.head

In [ ]:
#choose only the asia countries

asia_countries = [
    "Afghanistan", "Armenia", "Azerbaijan", "Bahrain", "Bangladesh", "Bhutan", "Brunei Darussalam",
    "Cambodia", "China", "Cyprus", "Georgia", "India", "Indonesia", "Iran, Islamic Rep.", "Iraq",
    "Israel", "Japan", "Jordan", "Kazakhstan", "Korea, Dem. Rep.", "Korea, Rep.", "Kuwait",
    "Kyrgyz Republic", "Lao PDR", "Lebanon", "Malaysia", "Maldives", "Mongolia", "Myanmar",
    "Nepal", "Oman", "Pakistan", "Philippines", "Qatar", "Saudi Arabia", "Singapore", "Sri Lanka",
    "Syrian Arab Republic", "Tajikistan", "Thailand", "Timor-Leste", "Turkey", "Turkmenistan",
    "United Arab Emirates", "Uzbekistan", "Vietnam", "Yemen, Rep.",
    "Hong Kong SAR, China", "Macao SAR, China"
]

asiaData = cleanData[cleanData["Country name"].isin(asia_countries)].copy()
asiaData

In [ ]:
#replace ".." with "NaN"
year_cols = [int(y) for y in range(1990, 2012)]
for i in year_cols:
  asiaData[i] = asiaData[i].replace("..", pd.NA)

asiaData

In [ ]:
#split the data into different variables based on their name
series_name = asiaData["Series name"].unique()
series_name_var = [
    "cereal_yield", "FDI", "energyGDP", "energyCap", "CO2", "GHG", "popUrban", "nationalTPA",
    "pavedR", "GDP", "GNI", "mortRate", "popGrowth", "pop", "urbanPopGrowth", "urbanPop"
    ]

for i in range(len(series_name)):
  var_name = series_name_var[i]
  series_value = series_name[i]
  globals()[var_name] = asiaData[asiaData["Series name"] == series_value].copy()
  print(f"{var_name} -> {series_value}")
asiaData.head()

In [ ]:
"""
clean data by removing the column of year if the column value is NA and using the mean value to fill in the data
here we use dictionary to help with the loop
"""

dataDict = {
    "cereal_yield": cereal_yield,
    "FDI": FDI,
    "energyGDP": energyGDP,
    "energyCap": energyCap,
    "CO2": CO2,
    "GHG": GHG,
    "popUrban": popUrban,
    "nationalTPA": nationalTPA,
    "pavedR": pavedR,
    "GDP": GDP,
    "GNI": GNI,
    "mortRate": mortRate,
    "pop": pop,
    "popGrowth": popGrowth,
    "urbanPopGrowth": urbanPopGrowth,
    "urbanPop": urbanPop
}

In [ ]:
dataDict["FDI"]

In [ ]:
"""
#fill by using mean
for name, data in dataDict.items():
  data = data.copy()
  data = data.dropna(axis=1, how="all")   # remove empty columns
  for col in year_cols:
    if col in data.columns:
      col_mean = data[col].mean(skipna=True)
      data[col] = data[col].fillna(col_mean)

  dataDict[name] = data

dataDict["pop"].head()
"""


In [ ]:
#fill by using INTERPOLATION
for name, data in dataDict.items():
    data = data.copy()

    #remove completely empty columns
    data = data.dropna(axis=1, how="all")

    #find only columns that are actually years in THIS dataset
    year_cols = [col for col in data.columns if str(col).strip().isdigit()]

    #if there are no year columns, skip this dataset
    if len(year_cols) == 0:
        print(f"Skipping {name}: no year columns found.")
        dataDict[name] = data
        continue

    #make sure they are numeric (so we can interpolate)
    data[year_cols] = data[year_cols].apply(pd.to_numeric, errors="coerce")

    #interpolate for each country (and series, if column exists)
    group_cols = ["Country name"]
    if "Series name" in data.columns:
        group_cols.append("Series name")

    def fill_group(df):
        #only keep existing year columns
        cols = [c for c in year_cols if c in df.columns]
        df[cols] = (df[cols].interpolate(method="linear", axis=1).ffill(axis=1).bfill(axis=1))
        return df

    data = data.groupby(group_cols, group_keys=False).apply(fill_group)

    #fill the missing row with mean value
    for col in year_cols:
      col_mean = data[col].mean(skipna=True)
      data[col] = data[col].fillna(col_mean)

    dataDict[name] = data

dataDict["FDI"]


In [ ]:
#move the year become column for CO2 variable

#CO2
df_CO2 = dataDict["CO2"]

#move all the year columns into a single "Year" column
df_CO2_long = df_CO2.melt(
    id_vars=["Country name", "Series name"],  #columns to keep fixed
    var_name="Year",                          #new column name for years
    value_name="Value_CO2"                    #new column name for values
)
#convert Year column to integer
df_CO2_long["Year"] = df_CO2_long["Year"].astype(int)
df_CO2_final = df_CO2_long[["Country name", "Year", "Value_CO2"]]
df_CO2_final



In [ ]:
#create new dictionary containing each variable values with CO2
import pandas as pd
import seaborn as sb

co2_vs_variable = {}  #new dictionary to store CO2 vs each variable

for var_name, df_var in dataDict.items():
    if var_name == "CO2":  #skip CO2 itself
        continue

    #move the variable dataframe into long format
    df_var_long = df_var.melt(
        id_vars=["Country name", "Series name"],
        var_name="Year",
        value_name=f"Value_{var_name}"
    )
    df_var_long["Year"] = df_var_long["Year"].astype(int)

    #Keep only Country_Year and value
    df_var_final = df_var_long[["Country name", "Year", f"Value_{var_name}"]]

    #Combine with CO2 values
    df_combined = pd.merge(df_var_final, df_CO2_final, on=["Country name", "Year"], how="inner")

    #drop rows with missing values cuz co2 only till 2008
    df_combined = df_combined.dropna(subset=[f"Value_{var_name}", "Value_CO2"])

    #store in dictionary
    co2_vs_variable[var_name] = df_combined

co2_vs_variable['nationalTPA']



In [ ]:
#create new data frame to combine all variable's value to do correlation
#start with CO2 values
all_values = df_CO2_final[["Value_CO2"]].copy()

#use loop over all other variables and append their values
for var_name, df_var in co2_vs_variable.items():
    #make sure the order matches by Country_Year
    all_values[f"{var_name}"] = df_var[f"Value_{var_name}"].values

#drop rows with any missing values
all_values = all_values.dropna()

all_values.corr()

# Heatmap of the Correlation Matrix
f, axes = plt.subplots(1, 1, figsize=(20, 20))
sb.heatmap(all_values.corr(), vmin = -1, vmax = 1, linewidths = 1,
           annot = True, fmt = ".2f", annot_kws = {"size": 18}, cmap = "RdBu")



In [ ]:
def calculate_r2_and_plot(df_series, var_name, all_year_cols):
    #Calculates the R^2 value for a linear trend over time using Asia Average data, and generates a plot showing the trend line. It extracts the series name
    #directly from the DataFrame (avoiding 'series_map').

    #Returns a dictionary with Series Name and R^2.

    #extract the full series name directly from the DataFrame (no series_map needed)
    series_name = df_series["Series name"].iloc[0] if not df_series.empty else "Unknown Series"

    #melt the DataFrame to long format
    df_long = pd.melt(
        df_series,
        id_vars=["Country name", "Series name"],
        value_vars=[col for col in all_year_cols if col in df_series.columns],
        var_name="Year",
        value_name="Value"
    )
    df_long['Year'] = pd.to_numeric(df_long['Year'])

    #calculate the average per year
    df_avg = df_long.groupby('Year', as_index=False)['Value'].mean()
    df_avg = df_avg.dropna()


    #perform Linear Regression
    X = df_avg[['Year']]
    Y = df_avg['Value']

    linreg = LinearRegression()
    linreg.fit(X, Y)
    Y_pred = linreg.predict(X)

    #calculate R-squared value
    r_squared = r2_score(Y, Y_pred)

    #plotting Preparation (for visualization if desired)
    df_avg['Country name'] = 'Asia Average'
    df_avg['Series name'] = series_name
    df_plot_data = pd.concat([df_long, df_avg], ignore_index=True)

    df_asia_avg = df_plot_data[df_plot_data['Country name'] == 'Asia Average']
    df_countries = df_plot_data[df_plot_data['Country name'] != 'Asia Average']

    #plotting
    plt.figure(figsize=(12, 6))

    #plot individual countries (light gray for context)
    for country in df_countries['Country name'].unique():
        country_data = df_countries[df_countries['Country name'] == country]
        plt.plot(
            country_data['Year'],
            country_data['Value'],
            color='lightgray',
            alpha=0.4,
            linewidth=1,
            zorder=1
        )

    #plot the Asia Average (Observed data)
    plt.plot(
        df_asia_avg['Year'],
        df_asia_avg['Value'],
        color='red',
        linewidth=3,
        label='Asia Average (Observed)',
        zorder=2
    )

    #plot the Linear Regression Line (Predicted data)
    plt.plot(
        df_asia_avg['Year'],
        Y_pred,
        color='blue',
        linestyle='--',
        linewidth=2,
        label=f'Linear Fit (R^2 = {r_squared:.4f})',
        zorder=3
    )

    #customize the plot
    plt.title(f'Time Series: {series_name}', fontsize=14)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel(series_name, fontsize=12)
    plt.ticklabel_format(style='plain', axis='y')
    plt.xticks(df_plot_data['Year'].unique().astype(int)[::2])
    plt.legend(loc='upper left')
    plt.show()
    plt.close()

    return {'Series Name': series_name, 'R^2': r_squared, 'Variable Name': var_name}

#iteration over the dataDict
r2_results = []

year_cols_str = [1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,
                 2003,2004,2005]

#the for loop structure to iterate over the dictionary:

newer_data = dataDict

selected_vars = ["GHG", "GDP", "urbanPop", "pop"]

for var_name, df_series in newer_data.items():
    if var_name in selected_vars:
        r2_data = calculate_r2_and_plot(df_series, var_name, year_cols_str)
        r2_results.append(r2_data)

print(r2_results)

In [ ]:
"""
#SAME ARIMA BUT THIS HARDCODED (1,1,1), which for some scenario wouldnt suit best
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

def add_arima_forecast_to_dataDict(dataDict, selected_vars, train_years, test_years, forecast_horizon=10):
    updated_dict = {}

    for var_name, df_series in dataDict.items():
        if var_name not in selected_vars:
            continue

        print(f"\n Processing {var_name}...")

        #identify year columns
        year_cols = [c for c in df_series.columns if str(c).isdigit()]
        year_cols = sorted(map(int, year_cols))

        #compute Asia average for each year
        asia_avg = df_series[year_cols].mean(axis=0)
        asia_avg = asia_avg.reset_index()
        asia_avg.columns = ["Year", "Value"]

        #split train/test
        train = asia_avg[asia_avg["Year"].isin(train_years)]
        test = asia_avg[asia_avg["Year"].isin(test_years)]

        if train.empty or test.empty:
            print(f"Skipping {var_name} (not enough data for train/test).")
            continue

        #fit ARIMA model
        y_train = train["Value"].values
        model = ARIMA(y_train, order=(1, 1, 1))
        model_fit = model.fit()

        #forecast test years
        n_test = len(test)
        y_pred_test = model_fit.forecast(steps=n_test)
        mse = mean_squared_error(test["Value"].values, y_pred_test)
        print(f"{var_name} → Test MSE = {mse:.4f}")

        #forecast next 10 years after last available year
        total_steps = n_test + forecast_horizon
        forecast_values = model_fit.forecast(steps=total_steps)

        last_train_year = train["Year"].max()
        future_years = [last_train_year + i for i in range(1, total_steps + 1)]

        #add forecast columns into DataFrame
        for i, year in enumerate(future_years[-forecast_horizon:]):  #only add the 10 years
            df_series[year] = forecast_values[n_test + i]  #asia average forecast for all countries

        updated_dict[var_name] = df_series

        #plot every graphs
        plt.figure(figsize=(10, 5))
        plt.plot(train["Year"], train["Value"], marker="o", label="Train Data")
        plt.plot(test["Year"], test["Value"], marker="o", color="green", label="Test Data (Actual)")
        plt.plot(test["Year"], y_pred_test, "--", color="orange", label="Test Forecast")
        plt.plot(future_years[-forecast_horizon:], forecast_values[-forecast_horizon:], "--", color="blue", label=f"Future Forecast (+{forecast_horizon} yrs)")
        plt.title(f"ARIMA Forecast for {var_name}\nTest MSE = {mse:.4f}")
        plt.xlabel("Year")
        plt.ylabel(var_name)
        plt.legend()
        plt.show()

    return updated_dict


#example usage
selected_vars = ["urbanPop", "pop", "GDP", "cereal_yield"]

train_years = list(range(1990, 2006))  #train: 1990–2005
test_years = [2006, 2007, 2008]        #test: 2006–2008

#add forecasts (next s10 years)
dataDict = add_arima_forecast_to_dataDict(dataDict, selected_vars, train_years, test_years, forecast_horizon=10)

#check updated columns
for var in selected_vars:
    print(f"\n{var} columns:", [c for c in dataDict[var].columns if str(c).isdigit()][-15:])
"""

In [ ]:
#better arima model than the above, since this one find the best combination to find the lowerst MSE
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import itertools

def add_arima_forecast_to_dataDict(dataDict, selected_vars, train_years, test_years, forecast_horizon=10):
    updated_dict = {}

    #search range for ARIMA(p,d,q)
    p_values = [4, 5, 6, 7]
    d_values = [1, 2, 3]
    q_values = [4, 5, 6, 7]

    for var_name, df_series in dataDict.items():
        if var_name not in selected_vars:
            continue

        #identify year columns, sort and conv to int
        year_cols = [c for c in df_series.columns if str(c).isdigit()]
        year_cols = sorted(map(int, year_cols))

        # compute Asia average for each year
        asia_avg = df_series[year_cols].mean(axis=0).reset_index()
        asia_avg.columns = ["Year", "Value"]

        # split train/test
        train = asia_avg[asia_avg["Year"].isin(train_years)]
        test = asia_avg[asia_avg["Year"].isin(test_years)]

        if train.empty or test.empty:
            continue

        y_train = train["Value"].values

        #find best (p,d,q) by finding lowest MSE on test data
        best_order, best_mse = None, float("inf")

        #iterate using itertools
        for p, d, q in itertools.product(p_values, d_values, q_values):
            try:
                model = ARIMA(y_train, order=(p, d, q))
                model_fit = model.fit()
                y_pred_test = model_fit.forecast(steps=len(test))
                mse = mean_squared_error(test["Value"].values, y_pred_test)
                if mse < best_mse:
                    best_mse = mse
                    best_order = (p, d, q)
            except Exception:
                continue

        print(f"best ARIMA order for {var_name}: {best_order} with MSE = {best_mse:.4f}")

        #final order aft iteration
        final_model = ARIMA(y_train, order=best_order)
        final_fit = final_model.fit()

        #predict
        total_steps = len(test) + forecast_horizon
        forecast_values = final_fit.forecast(steps=total_steps)
        last_train_year = train["Year"].max()
        future_years = [last_train_year + i for i in range(1, total_steps + 1)]

        #create copy
        df_out = df_series.copy(deep=True)

        #add forecast by slicing last 10 years to dict
        for i, year in enumerate(future_years[-forecast_horizon:]):
            df_out[year] = forecast_values[len(test) + i]

        updated_dict[var_name] = df_out

        #plot
        plt.figure(figsize=(9,5))
        plt.plot(train["Year"], train["Value"], "o-", label="Train", color="#1f77b4")
        plt.plot(test["Year"], test["Value"], "o-", label="Test (Actual)", color="#2ca02c")
        plt.plot(test["Year"], forecast_values[:len(test)], "--", label="Test Forecast", color="#ff7f0e")
        plt.plot(future_years[-forecast_horizon:], forecast_values[-forecast_horizon:], "--s",
                 label=f"Future Forecast (+{forecast_horizon} yrs)", color="#9467bd")
        plt.title(f"{var_name} — ARIMA{best_order} | Test MSE = {best_mse:.4f}", fontsize=13)
        plt.xlabel("Year")
        plt.ylabel(var_name)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

    return updated_dict

newDataDict = {}

selected_vars = ["urbanPop", "pop", "GHG", "GDP"]

train_years = list(range(1990, 2006))  #train: 1990–2005
test_years = [2006, 2007, 2008]        #test: 2006–2008

newdataDict = add_arima_forecast_to_dataDict(dataDict, selected_vars, train_years, test_years, forecast_horizon=10)



In [ ]:
newdataDict

MAKE THE CO2 x DATASET TABLE

In [ ]:
Var1 = co2_vs_variable['GDP']
Var2 = co2_vs_variable['pop']
Var3 = co2_vs_variable['GHG']
Var4 = co2_vs_variable['urbanPop']

In [ ]:
def SEPARATOR(A):
  # Ensure Year is numeric
  A["Year"] = pd.to_numeric(A["Year"], errors="coerce")

  # Split
  df_1990_2004 = A[(A["Year"] >= 1990) & (A["Year"] <= 2004)].copy()
  df_2005_plus = A[A["Year"] >= 2005].copy()

  return df_1990_2004, df_2005_plus

Var1_1, Var1_2 = SEPARATOR(Var1)
Var2_1, Var2_2 = SEPARATOR(Var2)
Var3_1, Var3_2 = SEPARATOR(Var3)
Var4_1, Var4_2 = SEPARATOR(Var4)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
linreg = LinearRegression()

def LINREG(VARIABLE_1, VARIABLE_2, COLUMN_1, COLUMN_2):
  X_train = pd.DataFrame(VARIABLE_1[COLUMN_1])
  Y_train = pd.DataFrame(VARIABLE_1[COLUMN_2])
  X_test = pd.DataFrame(VARIABLE_2[COLUMN_1])
  Y_test = pd.DataFrame(VARIABLE_2[COLUMN_2])

  #TRAIN DATA
  linreg.fit(X_train, Y_train)
  print('Intercept \t: b = ', linreg.intercept_)
  print('Coefficients \t: a = ', linreg.coef_)

  train_pred = linreg.predict(X_train)

  f = plt.figure(figsize=(10, 5))
  plt.scatter(X_train, Y_train)
  plt.plot(X_train, train_pred, color = "r", linewidth = 3)
  plt.xlabel(COLUMN_1)
  plt.ylabel(COLUMN_2)
  plt.show()

  print("Explained Variance (R^2) \t:", r2_score(Y_train, train_pred))
  print("Mean Squared Error (MSE) \t:", mean_squared_error(Y_train, train_pred))
  print()

  #TEST DATA
  test_pred = linreg.predict(X_test)

  f = plt.figure(figsize=(10, 5))
  plt.scatter(X_test, Y_test)
  plt.plot(X_test, test_pred, color = "r", linewidth = 3)
  plt.xlabel(COLUMN_1)
  plt.ylabel(COLUMN_2)
  plt.show()

  print("Explained Variance (R^2) \t:", r2_score(Y_test, test_pred))
  print("Mean Squared Error (MSE) \t:", mean_squared_error(Y_test, test_pred))
  print()

LINREG(Var1_1, Var1_2, 'Value_GDP', 'Value_CO2')
LINREG(Var2_1, Var2_2, 'Value_pop', 'Value_CO2')
LINREG(Var3_1, Var3_2, 'Value_GHG', 'Value_CO2')
LINREG(Var4_1, Var4_2, 'Value_urbanPop', 'Value_CO2')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

#combine top 5 variables with CO2
#create new df without screwing base df
all_var = df_CO2_final.copy()

#combine each variable's data with co2
for var in ["urbanPop", "pop", "GDP", "GHG"]:
    df_var = co2_vs_variable[var][["Country name", "Year", f"Value_{var}"]]
    all_var = pd.merge(all_var, df_var, on=["Country name", "Year"], how="inner")

all_var

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score


#split train and test dataset, 2005 onwards as test data
train_data = all_var[all_var["Year"] < 2006]
test_data  = all_var[all_var["Year"].between(2006, 2008)]

X_train = train_data[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_GHG"]]
y_train = train_data["Value_CO2"]
X_test = test_data[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_GHG"]]
y_test = test_data["Value_CO2"]

#linreg
linreg = LinearRegression()
linreg.fit(X_train, y_train)


In [ ]:
"""
#same code, but basically here split random

#split train and test dataset randomly with 25% as test data
X = all_var[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_cereal_yield"]]
y = all_var["Value_CO2"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)

#linreg
linreg = LinearRegression()
linreg.fit(X_train, y_train)
"""

In [ ]:
#draw the distribution of response
f, axes = plt.subplots(1, 3, figsize=(24, 6))
sb.boxplot(data = y_train, orient = "h", ax = axes[0])
sb.histplot(data = y_train, ax = axes[1])
sb.violinplot(data = y_train, orient = "h", ax = axes[2])

In [ ]:
#draw the distributions of all predictors
f, axes = plt.subplots(4, 3, figsize=(18, 40))

count = 0
for var in X_train:
    sb.boxplot(data = X_train[var], orient = "h", ax = axes[count,0])
    sb.histplot(data = X_train[var], ax = axes[count,1])
    sb.violinplot(data = X_train[var], orient = "h", ax = axes[count,2])
    count += 1
plt.savefig("my_plotaaa.png")

In [ ]:
sb.pairplot(data = all_var[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_GHG","Value_CO2"]])

In [ ]:
  #print out the coeff
coef = pd.DataFrame({
    "Predictors": X_train.columns,
    "Coefficients": linreg.coef_
}).sort_values(by="Coefficients", ascending=False)

print("Intercept:", linreg.intercept_)
print("\nCoefficients:\n", coef, "\n")

# predict test data
y_train_pred = linreg.predict(X_train)
y_test_pred = linreg.predict(X_test)

#plot true value vs predicted value
sb.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

#train plot
sb.scatterplot(x=y_train, y=y_train_pred, color="#4C72B0", alpha=0.6, ax=axes[0])
sb.lineplot(x=y_train, y=y_train, color="red", linewidth=2, ax=axes[0])
axes[0].set_title("Training Data: True vs Predicted CO₂", fontsize=16)
axes[0].set_xlabel("True CO₂ Emissions")
axes[0].set_ylabel("Predicted CO₂ Emissions")
axes[0].text(0.05, 0.9, f"R² = {r2_score(y_train, y_train_pred):.3f}",
             transform=axes[0].transAxes, fontsize=14, color="black")

#test plot
sb.scatterplot(x=y_test, y=y_test_pred, color="#55A868", alpha=0.6, ax=axes[1])
sb.lineplot(x=y_test, y=y_test, color="red", linewidth=2, ax=axes[1])
axes[1].set_title("Test Data (2005–2008): True vs Predicted CO₂", fontsize=16)
axes[1].set_xlabel("True CO₂ Emissions")
axes[1].set_ylabel("Predicted CO₂ Emissions")
axes[1].text(0.05, 0.9, f"R² = {r2_score(y_test, y_test_pred):.3f}",
             transform=axes[1].transAxes, fontsize=14, color="black")

plt.show()

#calculate r squared and MSE
print("Goodness of Fit:")
print(f"Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Train MSE: {mean_squared_error(y_train, y_train_pred):.3f}")
print()
print(f"Test R²: {r2_score(y_test, y_test_pred):.3f}")
print(f"Test MSE: {mean_squared_error(y_test, y_test_pred):.3f}")

In [ ]:
# Gradient Boosting Regressor

In [ ]:
#fit grad boost
gbr = GradientBoostingRegressor()
gbr.fit(X_train, y_train)

#importance
importance_df = pd.DataFrame({
    "Predictors": X_train.columns,
    "Importance": gbr.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Feature Importances:\n")
print(importance_df, "\n")

#predict
y_train_pred = gbr.predict(X_train)
y_test_pred = gbr.predict(X_test)

#set plot
sb.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

#train
sb.scatterplot(x=y_train, y=y_train_pred, color="#4C72B0", alpha=0.6, ax=axes[0])
sb.lineplot(x=y_train, y=y_train, color="red", linewidth=2, ax=axes[0])
axes[0].set_title("Training Data: True vs Predicted CO₂", fontsize=16)
axes[0].set_xlabel("True CO₂ Emissions")
axes[0].set_ylabel("Predicted CO₂ Emissions")
axes[0].text(0.05, 0.9, f"R² = {r2_score(y_train, y_train_pred):.3f}",
             transform=axes[0].transAxes, fontsize=14, color="black")

#test
sb.scatterplot(x=y_test, y=y_test_pred, color="#55A868", alpha=0.6, ax=axes[1])
sb.lineplot(x=y_test, y=y_test, color="red", linewidth=2, ax=axes[1])
axes[1].set_title("Test Data (2005–2008): True vs Predicted CO₂", fontsize=16)
axes[1].set_xlabel("True CO₂ Emissions")
axes[1].set_ylabel("Predicted CO₂ Emissions")
axes[1].text(0.05, 0.9, f"R² = {r2_score(y_test, y_test_pred):.3f}",
             transform=axes[1].transAxes, fontsize=14, color="black")

plt.show()

#R2 MSE
print("Goodness of Fit:")
print(f"Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Train MSE: {mean_squared_error(y_train, y_train_pred):.3f}")
print()
print(f"Test R²: {r2_score(y_test, y_test_pred):.3f}")
print(f"Test MSE: {mean_squared_error(y_test, y_test_pred):.3f}")


In [ ]:
#DO FOR PREDICTION

In [ ]:
#same as before, create new dictionary of co2 vs all other variables, but this dictionary is only for predict

co2_vs_variable_predict = {}

df_CO2_final = df_CO2.melt(
    id_vars=["Country name", "Series name"],
    var_name="Year",
    value_name="Value_CO2"
)
df_CO2_final["Year"] = df_CO2_final["Year"].astype(int)
df_CO2_final = df_CO2_final[["Country name", "Year", "Value_CO2"]]

for var_name, df_var in newdataDict.items():
    if var_name == "CO2":
        continue

    df_var_long = df_var.melt(
        id_vars=["Country name", "Series name"],
        var_name="Year",
        value_name=f"Value_{var_name}"
    )
    df_var_long["Year"] = df_var_long["Year"].astype(int)
    df_var_final = df_var_long[["Country name", "Year", f"Value_{var_name}"]]

    df_combined = pd.merge(df_var_final, df_CO2_final, on=["Country name", "Year"], how="left")

    co2_vs_variable_predict[var_name] = df_combined

co2_vs_variable_predict["urbanPop"]


In [ ]:
#create dataframe combined all from the dictionary

#start with a base var, here i choose urbanPop
base_var = "urbanPop"
all_var_predict = co2_vs_variable_predict[base_var][["Country name", "Year", f"Value_{base_var}", "Value_CO2"]].copy()


#merge the other 3 variables
for var in ["GDP", "GHG", "pop"]:
    a = co2_vs_variable_predict[var][["Country name", "Year", f"Value_{var}", "Value_CO2"]]
    all_var_predict = pd.merge(all_var_predict, a, on=["Country name", "Year", "Value_CO2"], how="left")

all_var_predict

In [ ]:
#take predict data from 2008 onwards
predict_data = all_var_predict[all_var_predict["Year"] > 2008]

X_predict = predict_data[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_GHG"]]

#here, since we use the same model made previously
linreg = LinearRegression()
linreg.fit(X_train, y_train)

In [ ]:
predict_data

In [ ]:
#LINREG
X_predict = predict_data[["Value_urbanPop", "Value_pop", "Value_GDP", "Value_GHG"]]
y_predict_future = linreg.predict(X_predict)

#past data 1990 - 2008
actual = df_CO2_final.groupby("Year")["Value_CO2"].mean().reset_index()
actual = actual[actual["Year"] <= 2008]

predicted = pd.DataFrame({
    "Year": predict_data["Year"],
    "CO2": y_predict_future
})

#combine both from 1990–2018
timeline = pd.concat([actual.rename(columns={"Value_CO2": "CO2"}), predicted]).sort_values("Year")

#plot
plt.figure(figsize=(10,6))

#past
plt.plot(actual["Year"], actual["Value_CO2"], color="#1f77b4", marker="o", linewidth=2, label="Observed CO2 (1990–2008)")

#prediction
plt.plot(predicted["Year"], predicted["CO2"], color="#2ca02c", marker="o", linewidth=2, label="Predicted CO2 (2009–2018)")

plt.title("CO2 Emissions Timeline (1990–2018) — Linear Regression Model", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Average CO2 Emissions")
plt.legend()
plt.show()


In [ ]:
#GRAD BOOSTING

X_train_grad = X_train.copy()
y_train_grad = y_train.copy()
X_test_grad = X_test.copy()
y_test_grad = y_test.copy()

gbr = GradientBoostingRegressor()
gbr.fit(X_train_grad, y_train_grad)

y_predict_future = gbr.predict(X_predict)

predicted = pd.DataFrame({"Year": predict_data["Year"],"CO2": y_predict_future})

plt.figure(figsize=(10,6))

#plot past
plt.plot(actual["Year"], actual["Value_CO2"], color="#1f77b4", marker="o", linewidth=2, label="Observed CO2 (1990–2008)")

#plot predicted (Gradient Boosting)
plt.plot(predict_data["Year"], y_predict_future, color="#2ca02c", marker="o", linewidth=2, label="Predicted CO2 (2009–2018, Gradient Boosting)")

plt.title("CO2 Emissions Timeline (1990–2018) using Gradient Boosting Model", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Average CO2 Emissions")
plt.legend()
plt.show()


In [ ]:
"""
Problem Stattement : Predict the CO2 Change for the Next 5 years Using Linear Regression and 50 years using Dynamic Regression

Missing Value : Ken

Test Data =
    "Cereal yield (kg per hectare)",#1
    "Foreign direct investment, net inflows (% of GDP)",#1
    "Urban population",#1
    "Paved roads (% of total roads)",#1'
    "Energy use per capita (kilograms of oil equivalent)",#1
    "Energy use per units of GDP (kg oil eq./$1,000 of 2005 PPP $)",#1'
    "GHG net emissions/removals by LUCF (MtCO2e)",#1''
    "Population in urban agglomerations >1million (%)",#1'
    "Nationally terrestrial protected areas (% of total land area)",#1
    "GDP ($)",#1
    "GNI per capita (Atlas $)",#1
    "Under-five mortality rate (per 1,000)",#1
    "Population growth (annual %)", #1
    "Population", #1
    "Urban population growth (annual %)", #1
}
}

Predicted Data =
    "CO2 emissions per capita (metric tons)",#1
    "CO2 emissions, total (KtCO2)",#1
    "CO2 emissions per units of GDP (kg/$1,000 of 2005 PPP $)"#1

Linear Regression + PLOT for 5 best predeictor and 5 worst predictor (use data from 1990 - 2011)
- bivariate for every test data with respect to each predicted data : Brandon
- multivariate : Shotaro

Grad Boosting + Comparing Result : Ken + Nico
"""